# FILE 02: TIỀN XỬ LÝ DỮ LIỆU & THIẾT KẾ DATA WAREHOUSE

File này tụi em tập trung xử lý hai phần chính:
1. Gom bảng và làm sạch dữ liệu gốc của Olist (xử lý trùng dòng, fill missing dữ liệu, tính lại các cột số ngày giao hàng và tổng tiền).
2. Tách dữ liệu thành các bảng Dimension và Fact theo mô hình Star Schema để chuẩn bị cho phần tính toán BUC Cube và gom cụm phía sau.

In [1]:
import os
import pandas as pd

print("--- BƯỚC 1: ĐỌC DỮ LIỆU THÔ ---")
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
print("Đã tải thành công toàn bộ các tập dữ liệu gốc.")

--- BƯỚC 1: ĐỌC DỮ LIỆU THÔ ---
Đã tải thành công toàn bộ các tập dữ liệu gốc.


## Gom cụm dữ liệu Payment và Review trước khi Merge
Để tránh tình trạng đơn hàng có nhiều hình thức thanh toán hoặc nhiều lượt đánh giá làm nhân dòng (trùng lặp dữ liệu vô lý khi merge), tụi em aggregate theo order_id trước:
- Với Payments: Tính tổng số tiền (`payment_value`) và lấy phương thức thanh toán phổ biến nhất.
- Với Reviews: Tính điểm đánh giá trung bình (`review_score`).

In [2]:
print("--- BƯỚC 2: TỔNG HỢP (AGGREGATE) PAYMENT VÀ REVIEW ---")
# Aggregate Payment để tránh nhân dòng
payment_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode().iloc[0] if not x.mode().empty else 'unknown')
).reset_index()

# Aggregate Review để tránh nhân dòng
review_agg = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean')
).reset_index()

print("Đã gom cụm dữ liệu payment và review thành công, đảm bảo tính duy nhất theo order_id.")

--- BƯỚC 2: TỔNG HỢP (AGGREGATE) PAYMENT VÀ REVIEW ---
Đã gom cụm dữ liệu payment và review thành công, đảm bảo tính duy nhất theo order_id.


## Tích hợp dữ liệu và Tiền xử lý (Tạo bảng final_sales)
Tiến hành gộp các bảng lại, xử lý ép kiểu ngày tháng và làm sạch dữ liệu:
- Tính `delivery_days` = Ngày khách nhận - Ngày đặt hàng. Các dòng bị âm do lỗi hệ thống sẽ chuyển về `NaN` và fill bằng median.
- Tính `total_amount` = giá sản phẩm + chi phí vận chuyển.
- Xử lý khuyết thiếu (missing value) cho các cột danh mục, phương thức thanh toán và điểm đánh giá theo đúng yêu cầu.

In [3]:
print("--- BƯỚC 3: MERGE DATA VÀ TIỀN XỬ LÝ (PREPROCESSING) ---")
# Kết hợp các bảng dữ liệu lại với nhau
df = pd.merge(orders, customers, on='customer_id', how='inner')
df = pd.merge(df, order_items, on='order_id', how='inner')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, payment_agg, on='order_id', how='left')
df = pd.merge(df, review_agg, on='order_id', how='left')

# Chuyển đổi định dạng thời gian sang datetime
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    df[c] = pd.to_datetime(df[c])

# Xóa dòng trùng lặp hoàn toàn
df = df.drop_duplicates()

# Tính toán các chỉ số mới
df['total_amount'] = df['price'] + df['freight_value']
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Tính số ngày giao hàng, sửa lỗi ngày âm và điền khuyết bằng median
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df.loc[df['delivery_days'] < 0, 'delivery_days'] = pd.NA
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())

# Lọc đúng và đủ 15 cột anh Phúc yêu cầu cho final_sales
final_cols = [
    'order_id', 'customer_id', 'customer_unique_id', 'product_id', 'seller_id',
    'order_purchase_timestamp', 'order_month', 'customer_state', 'seller_state',
    'product_category_name', 'payment_type', 'price', 'freight_value',
    'total_amount', 'review_score', 'delivery_days'
]
final_sales = df[final_cols].copy()

# Điền giá trị thiếu (missing values)
final_sales['product_category_name'] = final_sales['product_category_name'].fillna('unknown')
final_sales['payment_type'] = final_sales['payment_type'].fillna('unknown')
final_sales['review_score'] = final_sales['review_score'].fillna(final_sales['review_score'].median())

# Xuất file dữ liệu sạch final_sales.csv
os.makedirs('../data/processed', exist_ok=True)
final_sales.to_csv('../data/processed/final_sales.csv', index=False)
print("-> Đã xử lý xong và xuất file final_sales.csv thành công!")

--- BƯỚC 3: MERGE DATA VÀ TIỀN XỬ LÝ (PREPROCESSING) ---
-> Đã xử lý xong và xuất file final_sales.csv thành công!


## Thiết kế cấu trúc Star Schema cho Kho dữ liệu
Bóc tách từ bảng dữ liệu sạch để tạo các bảng Dim và Fact. Phần này tụi em đã chuẩn hóa lại khóa chính của các bảng Dimension dựa trên đúng ID của đối tượng (`customer_id`, `product_id`, `seller_id`) chứ không dùng chung `order_id` nữa để tránh lỗi mô hình:
- Bảng chiều: `dim_customer`, `dim_product`, `dim_seller`, `dim_payment`, `dim_date`.
- Bảng sự kiện: `fact_sales` chứa các khóa ngoại và các cột số liệu tính toán.

In [4]:
print("--- BƯỚC 4: TÁCH BẢNG CHO MÔ HÌNH STAR SCHEMA (DATA WAREHOUSE) ---")
os.makedirs('../data/warehouse', exist_ok=True)

# Tạo các bảng Dimension (Bảng chiều) dựa trên ID thực thể
dim_customer = final_sales[['customer_id', 'customer_unique_id', 'customer_state']].drop_duplicates()
dim_customer.to_csv('../data/warehouse/dim_customer.csv', index=False)

dim_product = final_sales[['product_id', 'product_category_name']].drop_duplicates()
dim_product.to_csv('../data/warehouse/dim_product.csv', index=False)

dim_seller = final_sales[['seller_id', 'seller_state']].drop_duplicates()
dim_seller.to_csv('../data/warehouse/dim_seller.csv', index=False)

dim_payment = final_sales[['payment_type']].drop_duplicates()
dim_payment.to_csv('../data/warehouse/dim_payment.csv', index=False)

dim_date = final_sales[['order_id', 'order_month']].drop_duplicates()
dim_date.to_csv('../data/warehouse/dim_date.csv', index=False)

# Tạo bảng Fact (Bảng sự kiện)
fact_sales = final_sales[[
    'order_id', 'customer_id', 'product_id', 'seller_id', 'order_purchase_timestamp', 
    'price', 'freight_value', 'total_amount', 'review_score', 'delivery_days'
]].drop_duplicates()
fact_sales.to_csv('../data/warehouse/fact_sales.csv', index=False)

print("-> Đã tạo và xuất toàn bộ các bảng Dim và Fact thành công vào thư mục data/warehouse/!")

--- BƯỚC 4: TÁCH BẢNG CHO MÔ HÌNH STAR SCHEMA (DATA WAREHOUSE) ---
-> Đã tạo và xuất toàn bộ các bảng Dim và Fact thành công vào thư mục data/warehouse/!
